In [182]:
import sympy as sp
from sympy import *
from IPython.display import display, Math

def print_math(str):
    display(Math(str))

Q1. Derive Jacobian matrix

In [183]:
# Symbols
rho, u, E ,gamma= sp.symbols('rho u E gamma')
U1, U2, U3 = sp.symbols('U1 U2 U3')
U_sym = sp.Matrix([U1, U2, U3])

# Pressure
# p = (gamma - 1) * (U3 - sp.Rational(1,2) * U2**2 / U1)
p=sp.symbols("p")
p_subs = {
    p: (gamma - 1) * (U3 - sp.Rational(1,2) * U2**2 / U1)
}

# Flux components
F1 = U2
F2 = U1 * (U2/U1)**2 + p
F3 = (U2/U1) * (U3 + p)

# Flux vector
F = sp.Matrix([F1, F2, F3])
F

Matrix([
[            U2],
[  p + U2**2/U1],
[U2*(U3 + p)/U1]])

In [184]:
A = F.subs(p_subs).jacobian(U_sym)
U_subs = {
    U1: rho,
    U2: rho*u,
    U3: rho*E
}

cv, T, a, R = sp.symbols('cv T a R')
thermo_subs = {
    E:cv*T + u**2/2,
    cv:R/(gamma - 1),
    T: a**2 / (gamma * R)
}

A = sp.simplify(A.subs(U_subs).subs(thermo_subs))

A

Matrix([
[                                                                  0,                                                              1,         0],
[                                                 u**2*(gamma - 3)/2,                                                  u*(3 - gamma), gamma - 1],
[u*(-2*a**2 + gamma**2*u**2 - 3*gamma*u**2 + 2*u**2)/(2*(gamma - 1)), (a**2 - gamma**2*u**2 + 5*gamma*u**2/2 - 3*u**2/2)/(gamma - 1),   gamma*u]])

In [185]:
U =  U_sym.subs(U_subs).subs(thermo_subs)
print_math("U =")
U

<IPython.core.display.Math object>

Matrix([
[                                    rho],
[                                  rho*u],
[rho*(a**2/(gamma*(gamma - 1)) + u**2/2)]])

Q2. Derive right eigenvectors Q_a = [r1 r2 r3]

In [186]:
eigenvalues = [val for val, _, _ in A.eigenvects()]
eigenvectors = [v for _, _, vs in A.eigenvects() for v in vs]

# rescaling and reording eigenvectors to be consistent with notes
r1= eigenvectors[0]*(1/eigenvectors[0][0])
r2= eigenvectors[2]*(1/eigenvectors[2][0]) 
r3= -eigenvectors[1]*(1/eigenvectors[1][0])

print("Expanded, unscaled by rho/(2a)")
print_math("Q_A=")
Matrix.hstack(r1, r2, r3)


Expanded, unscaled by rho/(2a)


<IPython.core.display.Math object>

Matrix([
[     1,                                                                1,                                                                -1],
[     u,                (2*a*gamma - 2*a + 2*gamma*u - 2*u)/(2*gamma - 2),               -(-2*a*gamma + 2*a + 2*gamma*u - 2*u)/(2*gamma - 2)],
[u**2/2, (2*a**2 + 2*a*gamma*u - 2*a*u + gamma*u**2 - u**2)/(2*gamma - 2), -(2*a**2 - 2*a*gamma*u + 2*a*u + gamma*u**2 - u**2)/(2*gamma - 2)]])

In [187]:
# collect and simplify entries of r1, r2, r3 
def clean_r(v):
    return Matrix([collect((simplify(entry)), [u**2/2,  a**2,u*a]) for entry in v])
# scale r2 and r3 by rho/(2a) to be consistent with notes
r1=clean_r(r1)
r2=clean_r(r2)*rho/(2*a)
r3=clean_r(r3)*rho/(2*a)
Qa = Matrix.hstack(r1, r2, r3)

print_math("Q_A=")
Qa

<IPython.core.display.Math object>

Matrix([
[     1,                                                           rho/(2*a),                                                           -rho/(2*a)],
[     u,                                                   rho*(a + u)/(2*a),                                                    rho*(a - u)/(2*a)],
[u**2/2, rho*(a**2 + a*u*(gamma - 1) + u**2*(gamma - 1)/2)/(2*a*(gamma - 1)), rho*(-a**2 + a*u*(gamma - 1) + u**2*(1 - gamma)/2)/(2*a*(gamma - 1))]])

Q4. Derive flux-vector splitting

In [188]:
l1, l2, l3 = sp.symbols('l1 l2 l3')

Lambda = sp.diag(l1, l2, l3)
lambda_subs = {
    l1: eigenvalues[0],
    l2: eigenvalues[1],
    l3: eigenvalues[2]
}

print_math("\\Lambda =" + sp.latex(Lambda.subs(lambda_subs)))
Qa_inv = Qa.inv()
print_math("Q_A^{-1}=" + sp.latex(Qa_inv))


<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [189]:
F_pm =Qa*Lambda*Qa_inv*U
F_pm_factored = Matrix([factor(collect(row, [l1, l2, l3])) for row in F_pm/(rho/(2*gamma))])
F_pm_factored[2] = collect((F_pm_factored[2])*2*(gamma-1), [l1, l2, l3])/(2*(gamma-1))
print_math("F^{\\pm}=\\frac{\\rho}{2\\gamma}" + sp.latex(F_pm_factored)) 


<IPython.core.display.Math object>

In [190]:
Lambda_QaInv_U= sp.simplify(Lambda*Qa_inv*U) 
print_math("F^{\\pm} = \\frac{1}{\\gamma}Q_A" + sp.latex(Lambda_QaInv_U))

print_math("F^{\\pm} = " + sp.latex(Lambda_QaInv_U[0]) + sp.latex(r1)+ "+" + sp.latex(Lambda_QaInv_U[1]) + sp.latex(r2) + "+" + sp.latex(Lambda_QaInv_U[2]) + sp.latex(r3))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Q4. Van Leer Flux Splitting

In [191]:
lambdas = sp.Matrix([l1, l2, l3])
M = sp.symbols('M')

In [192]:
# b makes derivations a bit cleaner
b = simplify(Lambda*Qa_inv*U) 
print_math("b = \\Lambda Q_a^{-1} U = " 
           + sp.latex(b) + "="+sp.latex(gamma)+ sp.latex(b*gamma)
           )
b_coeff = sp.Matrix([sp.collect(bi, li, evaluate=False)[li] 
                    for bi, li in zip(b, lambdas)])
D = sp.diag(*b_coeff)
# another way to express b
print_math("b = D"+sp.latex(lambdas) + "="+ 
    sp.latex(gamma)+ sp.latex(D*gamma)+  sp.latex(lambdas)
        )

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [193]:
print_math("F^{\\pm} = Q_a \\Lambda Qa^{-1} U  = Q_a b = Q_a D"+sp.latex(lambdas))
# introduce C, matrix for making it easier to write flux split
print_math(sp.latex(lambdas) +"=D^{-1} Q_a^{-1} F^{\\pm}")
print_math("\\text{Let } C = D^{-1} Q_a^{-1}")
C = D.inv()*Qa.inv()
# for C to be consistent with textbook format, needs to be factored
C_factor=gamma/(rho*a**2)
print_math(sp.latex(lambdas) +"= "
           +"CF^{\\pm}= "
           + sp.latex(C_factor)+sp.latex((C/C_factor).applyfunc(lambda x: sp.collect(sp.apart(x, gamma), [u**2/2,u,])))
           +"F^{\\pm}"
           )


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [211]:
# van leer fluxes from textbook
F_vl_p = (rho*a/4)*(M+1)**2 * sp.Matrix([
        1,
        ((gamma-1)*u+2*a)/gamma,
        ((gamma-1)*u+2*a)**2/(2*(gamma+1)*(gamma-1))
    ])

F_vl_m = -(rho*a/4)*(M-1)**2 * sp.Matrix([
        1,
        ((gamma-1)*u-2*a)/gamma,
        ((gamma-1)*u-2*a)**2/(2*(gamma+1)*(gamma-1))
    ])
print_math("F^{+} = " + sp.latex(F_vl_p) + "; ~F^{-} = " + sp.latex(F_vl_m))
# often factored out F_1
print_math("F^{+} = "+ sp.latex(F_vl_p[0]) + sp.latex((F_vl_p/F_vl_p[0])))
print_math("F^{-} = "+ sp.latex(F_vl_m[0]) + sp.latex((F_vl_m/F_vl_m[0])))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [212]:
lp = (C*F_vl_p).subs({u:M*a})
lp_factored =(lp/F_vl_p[0]).applyfunc(lambda x: sp.collect(x.simplify(),[M, gamma]))

lm = (C*F_vl_m).subs({u:M*a})
lm_factored =(lm/F_vl_m[0]).applyfunc(lambda x: sp.collect(x.simplify(),[M, gamma]))

for i,(lp_i, lm_i) in enumerate(zip(lp_factored, lm_factored)):
    print_math(f"\\lambda_{i+1}^+ =  "+ sp.latex(F_vl_p[0])+"~" +sp.latex(lp_i))
    print_math(f"\\lambda_{i+1}^- = "+ sp.latex(F_vl_m[0])+"~"+sp.latex(lm_i))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [223]:
# textbook solution
lp_vl = sp.Matrix([0,0,0])
lp_vl[0] = a/4*(M+1)**2 * ( 1 -(M-1)**2/(gamma+1))
lp_vl[1] = a/4*(M+1)**2 * ( 3 - M + (gamma-1)/(gamma+1)*(M-1)**2)
lp_vl[2] = a/2*(M+1)**2 *(M-1)/(gamma+1)*(1+(gamma-1)/2*M)

lm_vl = sp.Matrix([0,0,0])
lm_vl[0] = -lp_vl[0].subs({M:-M})
lm_vl[1] = -lp_vl[2].subs({M:-M})
lm_vl[2] = -lp_vl[1].subs({M:-M})

vl_lp_factored = sp.simplify(lp_vl/F_vl_p[0])
vl_lm_factored = sp.simplify(lm_vl/F_vl_m[0])


In [ ]:

items=zip(vl_lp_factored, vl_lm_factored,lp_factored, lm_factored)
print("checking equivalence (note these are factored):")
for i,(vl_lp_i, vl_lm_i,lp_i, lm_i) in enumerate(items):
    p_diff = (vl_lp_i-lp_i).simplify()
    print_math(f"\\lambda_{i+1}^+: ~" 
               +"~" + sp.latex(vl_lp_i) + "-" + sp.latex(lp_i)  
               +"=" + sp.latex(p_diff)) 
    m_diff = (vl_lm_i-lm_i).simplify()
    print_math(f"\\lambda_{i+1}^-: ~" 
               +"~" + sp.latex(vl_lm_i) + "-" + sp.latex(lm_i)  
               +"=" + sp.latex(m_diff))

checking equivalence (note these are factored)


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Derive F

In [225]:
# f1 = rho*u; mass flux
a1, b1, c1 = sp.symbols('a1 b1 c1')
a2, b2, c2 = sp.symbols('a2 b2 c2')
m1_p = a1*M**2 + b1*M + c1
m1_m = a2*M**2 + b2*M + c2
# f1 = (-1)*a*rho = 
# F+ moving to right
f1 = F[0].subs(U_subs).subs({u: M*a})
f1_p =  f1.subs({M: m1_p})
f1_m =  f1.subs({M: m1_m})

# 1. M = 1: fully supersonic to right
eq1 = sp.Eq(f1_p.subs(M, 1), f1.subs(M,1))  # f1+ @M=1 = f => m+ = 1
# 2. M = -1: fully supersonic to the left
eq2 = sp.Eq(f1_m.subs(M, -1), f1.subs(M,-1))  # f1-= @M=-1 = 0 => m- =0
# 3. M = -1: smoothness
eq3 = sp.Eq(sp.diff(f1_p, M).subs(M, -1), 0) # d(f1+)/dM @ M=-1 = 0 
# 4. M = 1: smoothness
eq4 = sp.Eq(sp.diff(f1_m, M).subs(M, 1), 0)  # d(f1-)/dM @ M=1 = 0 
# 5, 6.  conservation
eq5 = sp.Eq(a1+a2, 0)
eq6 = sp.Eq(b1+b2, 1)
sol = sp.solve([eq1, eq2, eq3, eq4, eq5, eq6], [a1, b1, c1, a2, b2, c2])
# sol = sp.solve([eq1, eq2, eq3], [a1, b1, c1])
f1_p = f1_p.subs(sol).factor()
f1_m = f1_m.subs(sol).factor()
print_math("f_1^+ = " + sp.latex(f1_p))
print_math("f_1^- = " + sp.latex(f1_m))
m1_p = m1_p.subs(sol).factor()
m1_m = m1_m.subs(sol).factor()    
print_math("M^+ = " + sp.latex(m1_p))
print_math("M^- = " + sp.latex(m1_m))


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [227]:
# f2 = rho*u**2 + p; momentum flux
p_expanded = p.subs(p_subs).subs(U_subs).subs(thermo_subs).simplify()

# f2 = rho*u**2 + p; momentum flux
a1, b1, c1, d1 = sp.symbols('a1 b1 c1 d1')
a2, b2, c2, d2 = sp.symbols('a2 b2 c2 d2')

f2 = F[1].subs(U_subs).subs({u: M*a})
m = f2-p

m2_p = m.subs({M**2: m1_p*(a1*M + b1)})
m2_m = m.subs({M**2: m1_m*(a2*M + b2)})
M_expr = collect(((m2_m + m2_p)/(a**2*rho)).expand().factor(),M)
print_math("\\frac{m^+ + m^-}{\\rho a^2}="+ sp.latex(M_expr) + "=M^2")
# 1. M**3
eq1 = sp.Eq((a1-a2)/4,0)
# 2. M**2
eq2 = sp.Eq((2*a1 + 2*a2 + b1 - b2)/4,1)
# 3. M
eq3 = sp.Eq((a1-a2+2*b1+2*b2)/4,0)
# 4. 0M
eq4 = sp.Eq((b1-b2)/4,0)

p_p = m1_p*(c1*M+d1)*p
p_m = m1_m*(c2*M+d2)*p
p_expr = sp.collect(((p_p+p_m)/p).factor(), M)
print_math("\\frac{p}{p}=\\frac{p^+ + p^-}{p} = " + sp.latex(p_expr) + "= 1")
# 5. M**3
eq5 = sp.Eq((c1-c2)/4, 0)
# 6. M**2
eq6 = sp.Eq((2*c1 + 2*c2 + d1 - d2)/4, 0)
# 7. M
eq7 = sp.Eq((c1 -c2 +2*d1 + 2*d2)/4, 0)
# 8. 0
eq8 = sp.Eq((d1-d2)/4, 1)

sol = sp.solve([eq1, eq2, eq3, eq4, eq5, eq6, eq7, eq8], [a1, b1, c1, d1, a2, b2, c2, d2])
print_math("sol="+sp.latex(sol))
m2_p = m2_p.subs(sol)
m2_m = m2_m.subs(sol)
p_p = p_p.subs(sol)
p_m = p_m.subs(sol)
f2_p = sp.simplify(((m2_p.subs(sol) + p_p.subs(sol))))
f2_m = sp.simplify(((m2_m.subs(sol) + p_m.subs(sol))))

f2_p_factored = sp.collect(((m2_p.subs(sol) + p_p.subs(sol))/f1_p).subs({M:u/a}).subs({p:p_expanded}).simplify(), u)
f2_m_factored = sp.collect(((m2_m.subs(sol) + p_m.subs(sol))/f1_m).subs({M:u/a}).subs({p:p_expanded}).simplify(), u)
print_math("f_2^+ = " + sp.latex(f2_p) +"=" + sp.latex(f1_p) +"\\cdot"+sp.latex(f2_p_factored))
print_math("f_2^- = " + sp.latex(f2_m)+"=" + sp.latex(f1_p) +"\\cdot"+sp.latex(f2_m_factored))


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>